# Predicting EV Purchases — A Data Scientist's Playbook

**Kaggle Playground S6E9** · binary classification · `Will_Buy_EV` ∈ {Yes, No}

This notebook is a **learning walkthrough**, not just a solution. Every section starts with *why we're doing this* before the code. The goal is to build the mental model of a working data scientist, so that on the next competition you already know what to reach for.

## The 8 phases we'll go through

1. **Frame the problem** — turn a business question into an ML task
2. **Set up a reliable evaluation** — this is the single most important step in Kaggle
3. **EDA** — look at the data with questions, not just plots
4. **Data quality & cleaning** — missing values, outliers, leakage
5. **Baseline model** — cheap, honest, unbeatable-until-it-is
6. **Feature engineering** — where domain thinking beats compute
7. **Model iteration** — try a few families, tune the promising one
8. **Ensemble + submit** — squeeze the last few points, then ship

A rule I want you to internalize: **at every stage, the question is "what would I need to see to convince myself this is real?"** Not "what's a cool thing to try."


---
## Phase 1 — Frame the problem

Before touching data, write down four things. If you can't, you don't understand the problem yet.

| Question | Answer for S6E9 |
|---|---|
| What are we predicting? | `Will_Buy_EV` — a single row per person, binary label |
| What's the granularity? | One row = one prospective buyer (identified by `id`) |
| What decision will this drive (hypothetically)? | Who to target with an EV marketing campaign / subsidy nudge |
| How will it be scored? | Kaggle competition page — usually **ROC-AUC** for binary playgrounds. **Verify this on the Evaluation tab before you submit.** The metric decides everything downstream. |

### Why the metric matters more than the model

- **AUC** — cares about ranking, not calibration. You can submit raw model scores. Class imbalance doesn't hurt you.
- **LogLoss** — cares about calibrated probabilities. You'll want `predict_proba` and often a calibration step.
- **F1 / accuracy** — cares about a threshold. You'll need to *tune the decision cutoff*, not just the model.

**Action for you:** open the competition's Evaluation tab, confirm the metric, and put it at the top of this notebook. The rest of the notebook assumes AUC — adjust if not.


## Phase 2 — Set up a reliable evaluation

**This is the single highest-leverage thing you can do in a Kaggle competition.** Everything else you try (features, models, tuning) is measured against your CV score. If your CV lies to you, every subsequent decision is noise.

### The rule: your CV split should mimic the train→test relationship

- Same class balance in each fold → **StratifiedKFold**
- Data has time order → **TimeSeriesSplit**
- Data has groups (same user in multiple rows) → **GroupKFold**
- None of the above (typical playground) → **StratifiedKFold** is the safe default

For S6E9 (one row per person, no time component), 5-fold Stratified is the right choice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)
SEED = 42
N_FOLDS = 5

# On Kaggle, these paths will be /kaggle/input/playground-series-s6e9/train.csv etc.
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'
SUB_PATH   = 'sample_submission.csv'
TARGET     = 'Will_Buy_EV'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)
print(train.shape, test.shape)

In [ ]:
# Encode target once, at the top, so we never confuse Yes/No vs 1/0 downstream.
y = (train[TARGET] == 'Yes').astype(int)
X = train.drop(columns=[TARGET, 'id'])
X_test = test.drop(columns=['id'])

# The CV splitter — reuse this OBJECT everywhere so folds are identical across experiments.
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print('Target rate:', y.mean().round(4))

### The one CV mistake that ruins everyone's first competition

**Fitting a transformer on the whole training set, then cross-validating.** Example: you scale features with `StandardScaler().fit(X)`, *then* run CV. The scaler saw the validation rows before they were held out. Your CV score is now optimistic, and it won't match the leaderboard.

The fix: put every transformer inside a `Pipeline`, so `.fit()` in each fold only sees that fold's training data. We'll do this consistently below.


## Phase 3 — EDA with questions, not plots

The trap in EDA: making 40 plots and knowing nothing more than before. Instead, walk in with **hypotheses** and let the data confirm or reject them.

For this dataset, my priors (before looking):

1. **Home_Charging_Possible = Yes** should be a huge positive signal. Charging is the #1 practical blocker for EV adoption.
2. **Range_Anxiety_Level** should be strongly negative, especially in Rural / high-commute segments.
3. **Subsidy_Available** should matter *conditional on income* (low-income more sensitive to subsidy).
4. **Environmental_Concern_Level** helps but is probably a weaker signal than the practical ones.
5. Age likely has a **non-monotonic** relationship: middle-aged people (with money + kids + practicality) buy more than the very young or the very old.

Writing priors down before looking keeps you honest — you can't retrofit "I always thought income would matter" once you see the correlation.


In [ ]:
# Schema at a glance — read this like a data dictionary
print(train.dtypes)
print('\nMissing values:')
print(train.isna().sum()[train.isna().sum() > 0])
print('\nCardinality of categoricals:')
for c in train.select_dtypes(include='object').columns:
    print(f'  {c}: {train[c].nunique()} unique — {train[c].unique()[:6]}')

In [ ]:
# Univariate: distribution of each numeric feature, colored by target.
# The question: does the CONDITIONAL distribution shift with the target?
num_cols = ['Age','Annual_Income_USD','Daily_Commute_km','Number_of_Cars_Owned',
            'Charging_Stations_Near_Home','Charging_Stations_Near_Work',
            'Environmental_Concern_Level']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.flat, num_cols):
    for lab, sub_ in train.groupby(TARGET):
        ax.hist(sub_[c].dropna(), bins=30, alpha=0.5, label=lab, density=True)
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Categoricals: buy-rate per level. This is your workhorse EDA chart for classification.
cat_cols = ['Gender','City_Type','Current_Car_Type','Home_Charging_Possible',
            'Subsidy_Available','Range_Anxiety_Level']

for c in cat_cols:
    rate = train.groupby(c)[TARGET].apply(lambda s: (s=='Yes').mean()).sort_values()
    n    = train.groupby(c).size()
    print(f'\n{c} (overall rate = {(y.mean()):.3f}):')
    for level, r in rate.items():
        print(f'  {level:20s} n={n[level]:6d}  buy_rate={r:.3f}  lift={r/y.mean():.2f}x')

### How to read a buy-rate table

- **Lift > 1.2 or < 0.8** with a healthy `n` → real signal, worth featuring on
- **Lift ≈ 1.0** → the level tells you nothing on its own — but might still matter in an *interaction*
- **Tiny `n` at some level** → noisy; be careful about over-fitting to it

Now revisit your priors. Which held up? Which were wrong? Being wrong here is the interesting part — that's where you learn the *actual* structure of the domain.

In [ ]:
# Interactions — often more predictive than any single feature.
# The classic one to check for EV: charging × commute distance.
pd.crosstab([train['Home_Charging_Possible'], pd.qcut(train['Daily_Commute_km'], 4)],
            train[TARGET], normalize='index').round(3)

## Phase 4 — Data quality & leakage

Quick sanity checks. A leaky feature can gift you a 0.99 CV that collapses on the leaderboard.

- **Impossible values?** e.g. `Age < 18` or `Annual_Income_USD < 0`
- **Duplicated rows?** `train.duplicated().sum()`
- **Test contains categories the train doesn't?** — will break one-hot encoders. Check with `set(test[c]) - set(train[c])`.
- **A feature that's a proxy for the target?** — if any single feature gets you AUC > 0.98 on its own, something is off.


In [ ]:
print('Duplicated rows:', train.duplicated().sum())
print('\nRanges:')
print(train[num_cols].describe().T[['min','max']])

print('\nUnseen test categories:')
for c in cat_cols:
    unseen = set(test[c].unique()) - set(train[c].unique())
    if unseen:
        print(f'  {c}: {unseen}')

# Single-feature AUC — a leakage sniffer
from sklearn.preprocessing import OrdinalEncoder
print('\nSingle-feature AUC (leakage sniff):')
for c in X.columns:
    x1 = X[[c]].copy()
    if x1[c].dtype == 'object':
        x1[c] = OrdinalEncoder().fit_transform(x1[[c]].fillna('NA'))
    x1 = x1.fillna(x1.median(numeric_only=True))
    print(f'  {c:32s} AUC={roc_auc_score(y, x1[c]):.3f}')

## Phase 5 — Baseline

Rules for a first baseline:

1. **As simple as possible.** Logistic regression on one-hot categoricals + median-imputed numerics.
2. **No tuning.** Defaults. The whole point is a floor to beat.
3. **Full pipeline.** So the CV score is honest and reusable.

If your first tree/gradient-boosting model can't beat this, something is broken (usually a bug, sometimes a bad CV split).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

cat = X.select_dtypes(include='object').columns.tolist()
num = X.select_dtypes(exclude='object').columns.tolist()

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore'))]), cat),
])

baseline = Pipeline([('pre', pre),
                     ('clf', LogisticRegression(max_iter=1000, random_state=SEED))])

scores = cross_val_score(baseline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'Baseline logreg AUC: {scores.mean():.4f} ± {scores.std():.4f}')

**Write this number down.** Every experiment from here on has to justify itself against it. Also note the **fold-to-fold std**: if it's high (>0.01 on AUC), your CV is noisy and small improvements will be indistinguishable from luck.

## Phase 6 — Feature engineering

**Feature engineering is applied EDA.** Every new feature should have a one-line rationale before you add it. "It might help" is not a rationale — that's just noise-fishing.

Candidates for this dataset, with rationale:

| Feature | Rationale |
|---|---|
| `charging_score = Charging_Stations_Near_Home + 0.5*Charging_Stations_Near_Work` | Access to charging is a single latent concept spread across two columns |
| `has_home_charging_and_stations = (Home_Charging_Possible=='Yes') & (Charging_Stations_Near_Home > median)` | Redundancy = certainty; both is a stronger positive than either |
| `cars_per_income = Number_of_Cars_Owned / log1p(Annual_Income_USD)` | Cars are a durable-goods signal; per-income normalizes for spending power |
| `commute_bucket = qcut(Daily_Commute_km, 4)` | Non-linear — a moderate commute may be ideal for an EV; too long triggers range anxiety |
| `age × EnvConcern` interaction | Older buyers with high concern may over-index; young low-concern likely doesn't buy |
| `income_bracket` | Buying a new car is a bracketed decision, not a linear one |

**Add features one at a time, re-run CV, keep only what helps.** Bulk-add will hide which idea was actually good.

In [ ]:
def add_features(df):
    df = df.copy()
    df['charging_score']  = df['Charging_Stations_Near_Home'] + 0.5 * df['Charging_Stations_Near_Work']
    df['cars_per_income'] = df['Number_of_Cars_Owned'] / np.log1p(df['Annual_Income_USD'].clip(lower=1))
    df['commute_bucket']  = pd.qcut(df['Daily_Commute_km'], q=4, labels=False, duplicates='drop')
    df['income_bracket']  = pd.qcut(df['Annual_Income_USD'], q=5, labels=False, duplicates='drop')
    df['age_x_env']       = df['Age'] * df['Environmental_Concern_Level']
    df['charging_and_home'] = ((df['Home_Charging_Possible']=='Yes') &
                               (df['Charging_Stations_Near_Home'] >
                                df['Charging_Stations_Near_Home'].median())).astype(int)
    return df

X_fe      = add_features(X)
X_test_fe = add_features(X_test)
print('New shape:', X_fe.shape)

### About target encoding

For high-cardinality categoricals, replacing each level with its mean target rate (**target encoding**) is powerful — and dangerous. If done wrong, it leaks. Do it *inside* the CV fold using `category_encoders.TargetEncoder` or a hand-rolled out-of-fold version. Skip it while categoricals stay small (this dataset qualifies).

## Phase 7 — Model iteration

For **tabular** data, the honest ordering of what usually wins:

1. Gradient-boosted trees — **LightGBM**, XGBoost, CatBoost. Almost always the strongest single model.
2. Logistic / linear regression with careful FE and regularization — great baseline, sometimes competitive.
3. Small MLP — occasionally adds diversity for an ensemble, rarely wins alone.
4. Random forest — fine but usually dominated by GBDT.

**Do not** start with a stacked neural net on a 100-column tabular problem. It will lose to a 30-line LightGBM script.

In [ ]:
import lightgbm as lgb

cat_cols_fe = X_fe.select_dtypes(include='object').columns.tolist()
for c in cat_cols_fe:
    X_fe[c]      = X_fe[c].astype('category')
    X_test_fe[c] = X_test_fe[c].astype('category')

def cv_lgb(params, X, y, cv, X_test=None):
    oof   = np.zeros(len(X))
    preds = np.zeros(len(X_test)) if X_test is not None else None
    for fold, (tr, va) in enumerate(cv.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y.iloc[tr],
                  eval_set=[(X.iloc[va], y.iloc[va])],
                  categorical_feature=cat_cols_fe,
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        if X_test is not None:
            preds += model.predict_proba(X_test)[:, 1] / cv.get_n_splits()
    print(f'OOF AUC: {roc_auc_score(y, oof):.4f}')
    return oof, preds

params = dict(
    n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=40,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=1.0,
    random_state=SEED, verbose=-1,
)
oof_lgb, pred_lgb = cv_lgb(params, X_fe, y, cv, X_test_fe)

### How to tune, without wasting a week on it

For LightGBM, the parameters that actually matter, in order:

1. `learning_rate` × `n_estimators` — you can leave `n_estimators` large and let early stopping pick it. Halve `learning_rate` for the final run.
2. `num_leaves` and `min_child_samples` — control tree complexity. Sweep `num_leaves ∈ {15, 31, 63, 127}` and `min_child_samples ∈ {10, 40, 100}`.
3. `subsample`, `colsample_bytree` — regularization by randomness. Try 0.7 / 0.8 / 0.9.
4. `reg_alpha`, `reg_lambda` — L1 / L2. Try 0, 1, 10.

**Use Optuna** for the search — but only after you have a working baseline pipeline and an honest CV. Tuning without those is polishing sand.

```python
import optuna
def objective(trial):
    p = dict(
        learning_rate=trial.suggest_float('lr', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('nl', 15, 127),
        min_child_samples=trial.suggest_int('mcs', 10, 200),
        subsample=trial.suggest_float('ss', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('cs', 0.6, 1.0),
        reg_lambda=trial.suggest_float('l2', 1e-3, 10, log=True),
        n_estimators=3000, random_state=SEED, verbose=-1,
    )
    oof, _ = cv_lgb(p, X_fe, y, cv)
    return roc_auc_score(y, oof)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
```

## Phase 8 — Ensemble & submit

The reliable playground recipe:

1. Train 2–3 **diverse** models — e.g. LightGBM + CatBoost + Logistic. Diversity of *errors* is what an ensemble monetizes; two GBDTs with the same params ensemble to themselves.
2. Average their **OOF predictions** first, and check that the ensemble's OOF AUC beats every single model. If not, don't ensemble.
3. Weight by OOF score (simple mean is fine when scores are close).
4. Only then average the **test-set predictions** using the same weights.

Averaging test predictions without checking OOF first is Kaggle Original Sin — you're guessing whether it helps.

In [ ]:
# Skeleton — add CatBoost / logistic OOFs the same way and average.
final_pred = pred_lgb  # replace with weighted average once you have >1 model
sub[TARGET] = np.where(final_pred >= 0.5, 'Yes', 'No')  # AUC only needs the score,
                                                        # but check the sample_submission format:
                                                        # if it wants probabilities, submit final_pred directly.
sub.to_csv('submission.csv', index=False)
print(sub.head())

---
## Post-submit — what to do with the leaderboard number

You'll see two AUCs now: **your CV** and **the public LB**. Compare them:

- **CV ≈ LB, small gap** → your CV is trustworthy. Iterate confidently.
- **CV >> LB** → you're leaking or overfitting. Suspect: target encoding, tuning too aggressively on CV, or a data-quality quirk in test.
- **CV << LB** → rare, but check for a bug in your OOF calculation.
- **CV varies a lot fold-to-fold** → increase folds (10), average seeds, or pick a better split.

Then keep a **leaderboard of your own experiments** in a spreadsheet: one row per submission, with CV, LB, and *what changed*. This is how you learn what actually moved the needle vs. what you talked yourself into.

---
## Habits worth internalizing from this project

1. Write your priors before you plot. Being wrong is the learning.
2. The pipeline (imputer → encoder → scaler → model) goes *inside* CV. Always.
3. Every new feature needs a one-sentence rationale.
4. Baseline first, tune last.
5. Trust your CV more than the public LB. The public LB is a noisy sample.
6. Keep an experiment log. Your future self will thank you.
